# Notebook 05 — Full Pipeline Demo

End-to-end demonstration of the two-phase Landslide Identification System:

| Phase | Model | Input | Output |
|-------|-------|-------|--------|
| 1 | AlexNet | Satellite/aerial image | Landslide / No Landslide + confidence |
| 2 | HMM | Historical disaster records | Type (Shallow/Deep/Debris/Rockfall) + probability |

**Prerequisites**: Run notebooks 00–04 first to train both models.

In [ ]:
import sys, os, json
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)
import config
import matplotlib.pyplot as plt
import cv2

## 1. Load Pipeline

In [ ]:
from pipeline.run_pipeline import LandslideIdentificationPipeline

pipeline = LandslideIdentificationPipeline(
    alexnet_checkpoint=config.ALEXNET_CHECKPOINT,
    hmm_model_path=config.HMM_MODEL_PATH,
    hmm_encoder_path=config.HMM_ENCODER_PATH,
)
print('Pipeline loaded successfully!')

## 2. Single Image Prediction Demo

In [ ]:
from pathlib import Path

# Pick 5 random test images (mix of landslide and non-landslide)
import random
random.seed(42)

test_images = []
for cls in ['landslide', 'non_landslide']:
    cls_dir = os.path.join(config.PROCESSED_DATA_DIR, 'test', cls)
    imgs = list(Path(cls_dir).glob('*.jpg')) + list(Path(cls_dir).glob('*.png'))
    if imgs:
        test_images += random.sample(imgs, min(3, len(imgs)))

print(f'Selected {len(test_images)} test images')

# Run predictions and display
fig, axes = plt.subplots(1, len(test_images), figsize=(5*len(test_images), 5))
if len(test_images) == 1:
    axes = [axes]

for ax, img_path in zip(axes, test_images):
    result = pipeline.predict(
        str(img_path),
        disaster_description='Debris slide',
        location='Northeast India',
        year=2022
    )

    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)

    phase1 = result['phase1']
    title = f"Label: {phase1['label']}\nConf: {phase1['confidence']:.2f}"
    if result['phase2']:
        title += f"\nType: {result['phase2']['landslide_type']}"
        title += f"\nProb: {result['phase2']['occurrence_probability']:.2f}"

    color = 'red' if phase1['label'] == 'landslide' else 'green'
    ax.set_title(title, fontsize=9, color=color)
    ax.axis('off')

plt.suptitle('Pipeline Predictions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Batch Predict on Full Test Set

In [ ]:
import pandas as pd

output_csv = os.path.join(PROJECT_ROOT, 'results', 'batch_predictions.csv')
os.makedirs(os.path.dirname(output_csv), exist_ok=True)

# Predict on all test landslide images
test_landslide_dir = os.path.join(config.PROCESSED_DATA_DIR, 'test', 'landslide')
df_results = pipeline.batch_predict(test_landslide_dir, output_csv)

print(f'\nBatch prediction complete: {len(df_results)} images')
print(f'Saved to: {output_csv}')
df_results.head(10)

In [ ]:
# Summary statistics
detected = df_results['phase1_label'].value_counts() if 'phase1_label' in df_results.columns else None
if detected is not None:
    print('Detection counts:')
    print(detected)
    correct_detections = (detected.get('landslide', 0) / len(df_results)) * 100
    print(f'\nLandslide detection rate: {correct_detections:.1f}%')

## 4. Full Evaluation Report

In [ ]:
report = pipeline.run_full_evaluation()

print('\n=== Performance Summary ===')
p1 = report['phase1']
print(f'Phase 1 (AlexNet):')
print(f'  Accuracy : {p1["accuracy"]:.4f}')
print(f'  Precision: {p1["precision"]:.4f}')
print(f'  Recall   : {p1["recall"]:.4f}')
print(f'  F1-Score : {p1["f1"]:.4f}')
p2 = report['phase2']
print(f'Phase 2 (HMM):')
print(f'  Log-likelihood: {p2["log_likelihood"]:.4f}')
print(f'  Records evaluated: {p2["n_records"]}')

## 5. End-to-End Scenario: New Image + Location Context

In [ ]:
# Simulate a new event: provide an image path and contextual information
# Replace 'path/to/your/image.jpg' with an actual image path

# Using one of our test images as example
example_image = str(list(Path(test_landslide_dir).glob('*.jpg'))[0])

result = pipeline.predict(
    image_path=example_image,
    disaster_description='Debris slide',      # Type of disaster description
    location='Meghalaya, Northeast India',
    duration=3,                                # 3-day event
    year=2022,
)

print('=== LANDSLIDE IDENTIFICATION RESULT ===')
print(f'Image: {os.path.basename(result["image_path"])}')
print()
print('PHASE 1 — Image Classification (AlexNet):')
print(f'  Label      : {result["phase1"]["label"].upper()}')
print(f'  Confidence : {result["phase1"]["confidence"]*100:.1f}%')
print(f'  Landslide probability    : {result["phase1"]["probabilities"]["landslide"]*100:.1f}%')
print(f'  Non-landslide probability: {result["phase1"]["probabilities"]["non_landslide"]*100:.1f}%')

if result['phase2']:
    p2 = result['phase2']
    print()
    print('PHASE 2 — Type Classification (HMM):')
    print(f'  Landslide Type     : {p2["landslide_type"]}')
    print(f'  Occurrence Probability: {p2["occurrence_probability"]*100:.1f}%')
    if p2.get('next_event_forecast'):
        f = p2['next_event_forecast'][0]
        print(f'  Next Event Forecast: {f["most_likely_state"]} (prob={f["probability"]*100:.1f}%)')

print()
print(f'FINAL VERDICT: {result["final_verdict"]}')